# HW2 Radar Workflow (Single Notebook)

This notebook replaces the multi-script workflow from the guide.

It follows the same assignment steps:
1. Download station measurements
2. Choose a 6-hour event window
3. Download radar RAW data
4. Unzip and check missing timestamps
5. Convert reflectivity to rainfall (with configurable Z-R)
6. Extract radar rainfall at station location
7. Compare measured vs radar rainfall (scatter + statistics)
8. Plot 6 hourly radar maps

In [ ]:
from pathlib import Path
import datetime as dt
import os
import shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import pearsonr

from scripts.measurement_download_parallel import fetch_data_for_parameters_parallel
from scripts.radar_download import download_radar_data_with_limit
import scripts.radar_unzip as radar_unzip
from scripts.radar_reflectivity_to_rainfall import main as process_radar_rainfall
from scripts.radar_extract import get_coords_arr, get_station_index
from scripts.radar_plot import plot_radar_polar

Path('data').mkdir(exist_ok=True)

## 0) Student Parameters

Update this cell for your own station/year and selected event.

In [ ]:
# --- Required choices from Moodle table ---
# --- Look up station coordinates from KAUR ---
# --- https://www.ilmateenistus.ee/meist/vaatlusvork/ --- 
STATION_NAME = 'Jõgeva'
STATION_COORDS = (58.749836, 26.415006)  # (lat, lon)

# Measurement search period (typically full assigned year)
MEAS_START = '2022-01-01 00:00:00'
MEAS_END = '2022-12-31 23:59:59'

# Z-R relationship: Z = a * R^b
ZR_A = 300.0
ZR_B = 1.5

# Radar download batching / API limits
INTERVAL_HOUR = 1
DAYS_PER_HOUR = 12

# Measurement downloader settings
MAX_WORKERS = 6
PARAMS_TO_DOWNLOAD = ['1h precipitation sum (mm)']

print('Station:', STATION_NAME)
print('Measurement range:', MEAS_START, 'to', MEAS_END)
print(f'Z-R: Z = {ZR_A} * R^{ZR_B}')

## 1) Download Measurements (Parallel)

In [ ]:
station_frames = fetch_data_for_parameters_parallel(
    params_to_download=PARAMS_TO_DOWNLOAD,
    stations_to_download=[STATION_NAME],
    start_date_str=MEAS_START,
    end_date_str=MEAS_END,
    max_workers=MAX_WORKERS,
)

if STATION_NAME not in station_frames:
    raise ValueError(f'No measurement dataframe returned for station {STATION_NAME}')

meas_df = station_frames[STATION_NAME].copy()
if 'PR1H' not in meas_df.columns:
    raise ValueError('Expected PR1H column missing. Check PARAMS_TO_DOWNLOAD.')

meas_csv = Path('data') / f"{STATION_NAME.replace(' ', '_')}_measurements.csv"
meas_df.to_csv(meas_csv)
print('Saved measurements to', meas_csv)

## 2) Inspect Measurements and Confirm Event Window

In [ ]:
plot_df = meas_df.copy()
plot_df.index = pd.to_datetime(plot_df.index)

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(plot_df.index, plot_df['PR1H'], color='tab:blue', linewidth=1.0, label='Measured PR1H')
#ax.axvspan(RADAR_START, RADAR_END, color='tab:red', alpha=0.2, label='Selected radar window')
ax.set_title(f'{STATION_NAME} hourly precipitation')
ax.set_xlabel('Datetime (UTC)')
ax.set_ylabel('Precipitation (mm)')
ax.grid(True, alpha=0.3)
ax.legend()
plt.show()

In [ ]:
# Ensure datetime index
work = meas_df.copy()
work.index = pd.to_datetime(work.index)

# Use hourly precipitation column
series = work["PR1H"].astype(float)

# 8-hour rolling std (window aligned to window end)
rolling_std_8h = series.rolling(window=8, min_periods=8).std()

# Find window with highest variability
best_end = rolling_std_8h.idxmax()
best_start = best_end - pd.Timedelta(hours=7)

# Radar download end can be set to xx:59
RADAR_START = best_start.to_pydatetime().replace(minute=0, second=0, microsecond=0)
RADAR_END = best_end.to_pydatetime().replace(minute=59, second=0, microsecond=0)

print(f"Highest variability 8h window std: {rolling_std_8h.max():.3f}")
print("RADAR_START:", RADAR_START)
print("RADAR_END:  ", RADAR_END)


In [ ]:
# Inspect top candidate windows
display(
    rolling_std_8h.dropna()
    .sort_values(ascending=False)
    .head(10)
    .rename("std_8h")
    .to_frame()
)

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(plot_df.index, plot_df['PR1H'], color='tab:blue', linewidth=1.0, label='Measured PR1H')
ax.set_title(f'{STATION_NAME} hourly precipitation')
ax.set_xlim(RADAR_START, RADAR_END)
ax.set_xlabel('Datetime (UTC)')
ax.set_ylabel('Precipitation (mm)')
ax.grid(True, alpha=0.3)
ax.legend()
plt.show()

## 2.1) Use more sofisticated search criteria

In [ ]:
s = meas_df["PR1H"].astype(float).copy()
s.index = pd.to_datetime(s.index)

W = 8  # hours

# Rolling features
feat = pd.DataFrame(index=s.index)
feat["std"] = s.rolling(W, min_periods=W).std()
feat["mean"] = s.rolling(W, min_periods=W).mean()
feat["max"] = s.rolling(W, min_periods=W).max()
feat["nonzero_frac"] = s.gt(0).rolling(W, min_periods=W).mean()   # % wet hours
feat["mid_frac"] = s.between(0.5, 5.0).rolling(W, min_periods=W).mean()  # in-between values

# Keep windows that look like "mixed rainfall", not all zeros and not only extremes
cand = feat.dropna().query(
    "nonzero_frac >= 0.5 and mid_frac >= 0.4 and mean >= 0.3 and max <= 12"
).copy()

# Score: high variability + many mid values + enough wet hours
cand["score"] = (
    0.45 * (cand["std"] / cand["std"].max()) +
    0.35 * cand["mid_frac"] +
    0.20 * cand["nonzero_frac"]
)

top = cand.sort_values("score", ascending=False).head(10)

# Build candidate start/end windows
candidates = []
for end_ts in top.index:
    start_ts = end_ts - pd.Timedelta(hours=W-1)
    candidates.append({
        "start": start_ts.floor("h"),
        "end": end_ts.replace(minute=59, second=0, microsecond=0),
        "score": top.loc[end_ts, "score"],
        "std": top.loc[end_ts, "std"],
        "mid_frac": top.loc[end_ts, "mid_frac"],
        "nonzero_frac": top.loc[end_ts, "nonzero_frac"],
    })

cand_df = pd.DataFrame(candidates)
display(cand_df)

In [ ]:
# function to plot candidates
def plot_candidate_windows(series, cand_df, n=6, pad_hours=2):
    """
    series: pandas Series with datetime index (e.g., PR1H)
    cand_df: DataFrame with columns ['start','end','score', ...]
    n: how many top candidates to plot
    """
    s = series.copy().astype(float)
    s.index = pd.to_datetime(s.index)

    show = cand_df.head(n).copy()

    for i, row in show.reset_index(drop=True).iterrows():
        start = pd.to_datetime(row["start"])
        end = pd.to_datetime(row["end"])

        seg = s.loc[start:end]
        if seg.empty:
            continue

        fig, ax = plt.subplots(figsize=(10, 3.2))
        ax.plot(seg.index, seg.values, marker="o", linewidth=1.2, color="tab:blue")
        ax.axvspan(start, end, color="tab:red", alpha=0.15)

        # highlight "in-between" precipitation values (0.5..5 mm)
        mid = seg[(seg >= 0.5) & (seg <= 5.0)]
        ax.scatter(mid.index, mid.values, color="tab:orange", s=35, zorder=3, label="0.5–5.0 mm")

        title = (
            f"Candidate #{i+1} | {start:%Y-%m-%d %H:%M} to {end:%Y-%m-%d %H:%M} | "
            f"score={row.get('score', float('nan')):.3f}"
        )
        ax.set_title(title)
        ax.set_ylabel("PR1H (mm)")
        ax.set_xlabel("Datetime (UTC)")
        ax.grid(True, alpha=0.3)
        ax.legend(loc="upper right")
        plt.tight_layout()
        plt.show()

In [ ]:
series = meas_df['PR1H']
plot_candidate_windows(series, cand_df, n=3)

## 3) Download Radar RAW Data for Selected 8-Hour Window

In [ ]:
# Selected 6-hour radar event window (after inspection)
RADAR_START = dt.datetime(2022, 8, 30, 4, 0)
RADAR_END = dt.datetime(2022, 8, 30, 11, 59)

In [ ]:
def clear_previous_radar_outputs(confirm=True):
    targets = [
        Path("data/radar_raw"),
        Path("data/radar_unzipped"),
        Path("data/radar_rainfall"),
    ]

    existing = [p for p in targets if p.exists() and any(p.iterdir())]
    if not existing:
        print("No existing radar output found. Nothing to delete.")
        return True

    print("Existing radar data found:")
    for p in existing:
        n = sum(1 for _ in p.rglob("*") if _.is_file())
        print(f" - {p} ({n} files)")

    if confirm:
        ans = input("Delete these folders and continue? Type 'YES' to confirm: ").strip()
        if ans != "YES":
            print("Deletion cancelled by user.")
            return False

    for p in existing:
        shutil.rmtree(p)
        p.mkdir(parents=True, exist_ok=True)

    print("Old radar data deleted. Fresh folders created.")
    return True

In [ ]:
# NB!  THIS TAKES TIME (8h took ~4 min)
assert RADAR_START < RADAR_END, 'RADAR_START must be before RADAR_END'

ok_to_continue = clear_previous_radar_outputs(confirm=True)

if ok_to_continue:
    
    download_radar_data_with_limit(
        start_datetime=RADAR_START,
        end_datetime=RADAR_END,
        interval_hour=INTERVAL_HOUR,
        days_per_hour=DAYS_PER_HOUR,
        raw=True,
    )

## 4) Unzip Radar Files and Check Missing Timestamps

In [ ]:
radar_unzip.start_date = RADAR_START
radar_unzip.end_date = RADAR_END

extracted_files = radar_unzip.extract_all_zips()
if not extracted_files:
    extracted_files = set(os.listdir(radar_unzip.OUTPUT_DIR))

actual_timestamps = radar_unzip.extract_timestamps_from_files(extracted_files)
expected_timestamps = radar_unzip.generate_expected_timestamps()

status = [{'Timestamp': ts, 'Available': ts in actual_timestamps} for ts in expected_timestamps]
missing_df = pd.DataFrame(status)
missing_csv = Path('data') / 'missing_data_final.csv'
missing_df.to_csv(missing_csv, index=False)

print('Saved missing-timestamp report to', missing_csv)
display(missing_df.head())
print('Available frames:', int(missing_df['Available'].sum()), '/', len(missing_df))

## 5) Convert Radar Reflectivity to Rainfall (Configurable Z-R)

In [ ]:
process_radar_rainfall(a=ZR_A, b=ZR_B, intervals=(1,))

## 6) Extract Radar Rainfall at Station Coordinates

In [ ]:
azims = np.load('data/radar_rainfall/rainfall_intensities/azimuths.npy')
ranges = np.load('data/radar_rainfall/rainfall_intensities/ranges.npy')
meta = np.load('data/radar_rainfall/rainfall_intensities/radar_metadata.npy').astype(float)

radar_lat = float(meta[0])
radar_lon = float(meta[1])

latarr, lonarr = get_coords_arr(ranges, azims, radar_lat, radar_lon)
lat_ind, lon_ind = get_station_index(latarr, lonarr, STATION_COORDS)

accum_dir = Path('data/radar_rainfall/accumulated_rainfall/1h')
rows = []
for fp in sorted(accum_dir.glob('rainfall_*.npy')):
    ts_str = fp.stem.split('_')[1]
    ts = dt.datetime.strptime(ts_str, '%Y%m%d%H%M')
    arr = np.load(fp)
    rows.append((ts, float(arr[lat_ind, lon_ind])))

radar_df = pd.DataFrame(rows, columns=['datetime', 'radar_rain_amount'])
radar_df.set_index('datetime', inplace=True)
radar_df.sort_index(inplace=True)

radar_csv = Path('data') / 'radar_rain_amount.csv'
radar_df.to_csv(radar_csv)
print('Saved station radar rainfall to', radar_csv)
display(radar_df.head())

## 7) Compare Radar vs Measured Rainfall

In [ ]:
meas_hourly = meas_df[['PR1H']].copy()
meas_hourly.index = pd.to_datetime(meas_hourly.index)

comparison = meas_hourly.join(radar_df, how='inner')
comparison = comparison.rename(columns={'PR1H': 'measured_rain_amount'})
comparison = comparison.dropna(subset=['measured_rain_amount', 'radar_rain_amount'])

if len(comparison) < 2:
    raise ValueError('Need at least 2 aligned points for statistics. Choose a different event window.')

x = comparison['measured_rain_amount'].to_numpy(dtype=float)
y = comparison['radar_rain_amount'].to_numpy(dtype=float)

pearson_r = pearsonr(x, y).statistic
rmsd = float(np.sqrt(np.mean((y - x) ** 2)))

def haversine_km(lat1, lon1, lat2, lon2):
    r = 6371.0
    dlat = np.radians(lat2 - lat1)
    dlon = np.radians(lon2 - lon1)
    a = np.sin(dlat / 2) ** 2 + np.cos(np.radians(lat1)) * np.cos(np.radians(lat2)) * np.sin(dlon / 2) ** 2
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))
    return r * c

distance_km = float(haversine_km(radar_lat, radar_lon, STATION_COORDS[0], STATION_COORDS[1]))

print(f'Aligned points: {len(comparison)}')
print(f'Pearson correlation: {pearson_r:.3f}')
print(f'RMSD (mm): {rmsd:.3f}')
print(f'Distance station-radar (km): {distance_km:.2f}')
display(comparison)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(comparison['measured_rain_amount'], comparison['radar_rain_amount'], color='tab:blue', s=45)

max_v = float(max(comparison['measured_rain_amount'].max(), comparison['radar_rain_amount'].max(), 1.0))
ax.plot([0, max_v], [0, max_v], linestyle='--', color='gray', linewidth=1.0, label='1:1 line')

ax.set_xlabel('Measured rainfall (mm)')
ax.set_ylabel('Radar rainfall (mm)')
ax.set_title('Measured vs Radar Rainfall')
ax.grid(True, alpha=0.3)
ax.legend()
plt.show()

## 8) Plot 6 Hourly Radar Maps

In [ ]:
plot_dir = Path('data/radar_plots')
plot_dir.mkdir(parents=True, exist_ok=True)

for timestamp in comparison.index:
    ts_key = timestamp.strftime('%Y%m%d%H%M')
    arr_path = Path('data/radar_rainfall/accumulated_rainfall/1h') / f'rainfall_{ts_key}.npy'
    if not arr_path.exists():
        print('Missing rainfall array:', arr_path.name)
        continue

    arr = np.load(arr_path)
    out_png = plot_dir / f'radar_{ts_key}_{STATION_NAME}.png'
    plot_radar_polar(
        array_to_plot=arr,
        title=f'Radar accumulated rainfall {ts_key}',
        filename=str(out_png),
        ranges=ranges,
        azimuths=azims,
        radar_lat=radar_lat,
        radar_lon=radar_lon,
        station_coords=STATION_COORDS,
        station_label='Jõgeva'
    )
    print('Saved', out_png)